In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# models 
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Activation, Rescaling, Flatten, Input, Concatenate
from tensorflow.keras.utils import to_categorical

## Preparing data

## Model build

## Sequential model vs functional model

## Pytania
- zwykłe warstwy jak są klasyfikowane te metody 
- funkcje aktywacji, można podawać od razu, 
- rozumiem że pomiędzy warstwami wymiary muszą się zgadzać ? 
- gdzie tutaj używa się tych metryk funkcji kosztu, ???


### Model build with 2D grey image input

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train.shape

In [ ]:
model = Sequential([
    Rescaling(1./255, input_shape=(28, 28)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='softmax'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    x_train, y_train,
    epochs=10, # number of times when model will go through the whole dataset
    batch_size=32, # amount of samples that will be used to calculate the gradient each time (data amount / batch_size = number of steps per epoch)
    validation_split=0.2 # it says we take 20% of training data to validate the model (check scores during training)
)

In [ ]:
# 6. Ewaluacja
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

### Functional model


In [ ]:

def build_model():
    # creating input layer
    inputs = Input(shape=(28, 28))
    # creating layers
    x = Rescaling(1./255)(inputs)  # normalizacja
    x = Flatten()(x)  # spłaszczenie danych
    x1 = Dense(64, activation='relu')(x)
    x2 = Dense(64, activation='relu')(x)
    merged = Concatenate()([x1, x2])
    # creating output
    outputs = Dense(4, activation='softmax')(merged)
    return Model(inputs=inputs, outputs=outputs)

### Testing optimize functions

####

#### Results
##### mnist data set
**4 classes** 
Sparse CCE           | Loss: 0.0285 | Accuracy: 0.9913
Categorical CCE      | Loss: 0.0196 | Accuracy: 0.9916
Hinge                | Loss: 0.7559 | Accuracy: 0.9899
MSE                  | Loss: 0.0028 | Accuracy: 0.9925
**6 classes**
Sparse CCE           | Loss: 0.0229 | Accuracy: 0.9918
Categorical CCE      | Loss: 0.0210 | Accuracy: 0.9937
Hinge                | Loss: 0.7560 | Accuracy: 0.9889
MSE                  | Loss: 0.0036 | Accuracy: 0.9901


In [ ]:
# 0. We reduce data size (for faster computation) & creating 
max_class = 6
mask = y_train < max_class
x_train = x_train[mask]
y_train = y_train[mask]

mask = y_test < max_class
x_test = x_test[mask]
y_test = y_test[mask]

# Wersja one-hot (do categorical_crossentropy)
y_train_oh = to_categorical(y_train, num_classes=4)
y_test_oh = to_categorical(y_test, num_classes=4)

losses = {
    "Sparse CCE": ("sparse_categorical_crossentropy", y_train, y_test),
    "Categorical CCE": ("categorical_crossentropy", y_train_oh, y_test_oh),
    "Hinge": ("hinge", y_train_oh, y_test_oh),
    "MSE": ("mean_squared_error", y_train_oh, y_test_oh)
}

results = {}

# training models
for name, (loss_fn, y_tr, y_te) in losses.items():
    model = build_model()
    model.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])
    print(f"\n🔧 Trening z funkcją kosztu: {name}")
    model.fit(x_train, y_tr, epochs=5, batch_size=64, verbose=0)
    loss, acc = model.evaluate(x_test, y_te, verbose=0)
    results[name] = (loss, acc)

In [ ]:
# for 4 classes
for name, (loss, acc) in results.items():
    print(f"{name:<20} | Loss: {loss:.4f} | Accuracy: {acc:.4f}")

In [ ]:
# for 6 classes 
for name, (loss, acc) in results.items():
    print(f"{name:<20} | Loss: {loss:.4f} | Accuracy: {acc:.4f}")
